In [8]:
!pip install torch
!pip install numpy
!pip install open_clip-torch
!pip install Pillow
!pip install scikit-learn
!pip install torch pandas

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import torch
import numpy as np
import random
import open_clip
from PIL import Image
from sklearn.model_selection import KFold
import pandas as pd
import glob

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Setup Model & Data
model_id = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BiomedCLIP model on {device}...")

model, _, preprocess_val = open_clip.create_model_and_transforms(model_id)
tokenizer = open_clip.get_tokenizer(model_id)
model.eval()
model.to(device)

# Automatically fetch all .png files from the directory
image_paths = np.array(sorted(glob.glob("/content/drive/MyDrive/Heartbeat_noColour/Heartbeat_noColour/At risk/*.png")))
image_paths2 = np.array(sorted(glob.glob("/content/drive/MyDrive/Heartbeat_noColour/Heartbeat_noColour/No risk/*.png")))

# Define Label Sets
no_colour_labels = [
    "a T wave with a high amount of blue/green",
    "a T wave with a high amount of orange/red",
]

# Pre-encode no_colour features before doing any predictions
text_c = tokenizer(no_colour_labels).to(device)

with torch.no_grad():
    text_features_c = model.encode_text(text_c)
    text_features_c /= text_features_c.norm(dim=-1, keepdim=True)

# Dictionary containing 4 distinct label sets
morphology_label_sets = {
    "No Label": [
        "Normal",
        "Abnormal"
    ],
    "L1": [
        "Normal T wave",
        "Abnormal T wave"
    ],
    "L2": [
        "Normal",
        "Flat",
        "Large",
        "Inverted",
        "Biphasic",
        "Notched"
    ],
    "L3": [
        "Normal T wave",
        "Abnormal T wave with flat amplitude",
        "Abnormal large T wave, taller than initial signal",
        '''Abnormal large T wave, broader and larger than
         initial signal''',
        "Abnormal T wave with a minor inversion",
        "Abnormal T wave with a deep inversion",
        "Abnormal T wave with a giant inversion",
        "Abnormal Biphasic T wave(Type A)",
        "Abnormal Biphasic T wave(Type B)",
        "Abnormal Bifid or Notched T wave"
    ]
}

# Helper Function for Inference
def evaluate_image(image_path, text_features_m, text_features_c, morphology_labels, alpha=0.5):

    # Pee-process images
    raw_image = Image.open(image_path).convert('RGB')
    image = preprocess_val(raw_image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        logit_scale = model.logit_scale.exp()

        # Logits and basic Probs
        logits_per_image_m = logit_scale * image_features @ text_features_m.t()
        logits_per_image_c = logit_scale * image_features @ text_features_c.t()

        probs_m = logits_per_image_m.softmax(dim=-1).cpu().numpy()[0]
        probs_c = logits_per_image_c.softmax(dim=-1).cpu().numpy()[0]

        # Fusion (Both)
        p_blue_green = probs_c[0]
        p_orange_red = probs_c[1]

        weight_blue_green = (p_blue_green * alpha) + (1.0 - alpha)
        weight_orange_red = (p_orange_red * alpha) + (1.0 - alpha)

        mixed_probs_m = []
        for i, label in enumerate(morphology_labels):
            weight = weight_blue_green if "Normal" in label else weight_orange_red
            mixed_probs_m.append(probs_m[i] * weight)

        mixed_probs_m = np.array(mixed_probs_m)
        mixed_probs_m /= mixed_probs_m.sum()

        # NONE (Baseline uniform distribution)
        none_probs = np.ones(len(morphology_labels)) / len(morphology_labels)

    return probs_m, probs_c, mixed_probs_m, none_probs


Loading BiomedCLIP model on cuda...


In [11]:
import glob
import numpy as np

# Define real image paths
high_risk_dir = "/content/drive/MyDrive/Heartbeat_noColour/Heartbeat_noColour/At risk/*.png"
no_risk_dir = "/content/drive/MyDrive/Heartbeat_noColour/Heartbeat_noColour/No risk/*.png"

# Retrieve and sort .png files
image_paths_abnormal = np.array(sorted(glob.glob(high_risk_dir)))
image_paths_normal = np.array(sorted(glob.glob(no_risk_dir)))

# Create ground truth labels
ground_truth_abnormal = np.array(['Abnormal'] * len(image_paths_abnormal))
ground_truth_normal = np.array(['Normal'] * len(image_paths_normal))

# Combine arrays
image_paths = np.concatenate((image_paths_abnormal, image_paths_normal))
ground_truth = np.concatenate((ground_truth_abnormal, ground_truth_normal))

# Print counts
print(f"Found {len(image_paths_abnormal)} high-risk (abnormal) images.")
print(f"Found {len(image_paths_normal)} no-risk (normal) images.")
print(f"Total combined images: {len(image_paths)}")
print(f"Total ground truth labels: {len(ground_truth)}")

Found 181 high-risk (abnormal) images.
Found 4573 no-risk (normal) images.
Total combined images: 4754
Total ground truth labels: 4754


In [12]:
import pandas as pd
import numpy as np
import torch
import os

real_inference_results_no_colour = []
desired_label_set_order = ["No Label", "L1", "L2", "L3"]

print("Starting inference loop including Colour Strategy...")
for label_set_name in desired_label_set_order:
    morph_labels = morphology_label_sets[label_set_name]

    # Pre-encode text features for the current morphology labels
    text_m = tokenizer(morph_labels).to(device)
    with torch.no_grad():
        text_features_m = model.encode_text(text_m)
        text_features_m /= text_features_m.norm(dim=-1, keepdim=True)

    # Iterate through all real image paths
    for path, true_label in zip(image_paths, ground_truth):
        try:
            # Evaluate image
            p_morph, p_col, p_both, p_none = evaluate_image(
                path, text_features_m, text_features_c, morph_labels, alpha=0.5
            )

            # Extract predicted labels
            pred_morph = morph_labels[np.argmax(p_morph)]
            pred_both = morph_labels[np.argmax(p_both)]
            pred_none = morph_labels[np.argmax(p_none)]

            # Colour Strategy mapping: blue/green (index 0) -> Normal, orange/red (index 1) -> Abnormal
            pred_col = 'Normal' if np.argmax(p_col) == 0 else 'Abnormal'

            # Extract file name from path
            file_name = os.path.basename(path)

            # Morphology Strategy
            real_inference_results_no_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_morph,
                'Strategy': 'No Colour + Morphology Context',
                'Label_Set': label_set_name
            })

            # Mixed Strategy
            real_inference_results_no_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_both,
                'Strategy': 'Colour + Morphology Context',
                'Label_Set': label_set_name
            })

            # Random Strategy
            real_inference_results_no_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_none,
                'Strategy': 'No Colour + No Morphology Context',
                'Label_Set': label_set_name
            })

            # Colour Strategy
            real_inference_results_no_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_col,
                'Strategy': 'Colour + No Morphology Context',
                'Label_Set': label_set_name
            })

        except Exception as e:
            print(f"Error processing {path}: {e}")

# Convert results to DataFrame
df_real_inference_no_colour = pd.DataFrame(real_inference_results_no_colour)
print("\nInference including Colour Strategy complete! First few results:")
display(df_real_inference_no_colour.head(10))

Starting inference loop including Colour Strategy...

Inference including Colour Strategy complete! First few results:


,File_Name,True_Label,Predicted_Label,Strategy,Label_Set
0,1415_Heartbeat_1.png,Abnormal,Abnormal,No Colour + Morphology Context,No Label
1,1415_Heartbeat_1.png,Abnormal,Abnormal,Colour + Morphology Context,No Label
2,1415_Heartbeat_1.png,Abnormal,Normal,No Colour + No Morphology Context,No Label
3,1415_Heartbeat_1.png,Abnormal,Abnormal,Colour + No Morphology Context,No Label
4,1426_Heartbeat_1.png,Abnormal,Abnormal,No Colour + Morphology Context,No Label
5,1426_Heartbeat_1.png,Abnormal,Abnormal,Colour + Morphology Context,No Label
6,1426_Heartbeat_1.png,Abnormal,Normal,No Colour + No Morphology Context,No Label
7,1426_Heartbeat_1.png,Abnormal,Abnormal,Colour + No Morphology Context,No Label
8,1441_Heartbeat_1.png,Abnormal,Abnormal,No Colour + Morphology Context,No Label
9,1441_Heartbeat_1.png,Abnormal,Abnormal,Colour + Morphology Context,No Label


In [13]:
import pandas as pd
import numpy as np
import math
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Map detailed predictions to binary
def map_to_binary(label):
    if 'normal' in label.lower() and 'abnormal' not in label.lower():
        return 'Normal'
    elif 'normal' in label.lower() and label.lower().startswith('normal'):
         return 'Normal'
    else:
        return 'Abnormal'

df_real_inference_no_colour['Binary_Prediction'] = df_real_inference_no_colour['Predicted_Label'].apply(map_to_binary)

# Map to numeric binary for sklearn metrics
df_real_inference_no_colour['True_Binary_Num'] = (df_real_inference_no_colour['True_Label'] == 'Abnormal').astype(int)
df_real_inference_no_colour['Pred_Binary_Num'] = (df_real_inference_no_colour['Binary_Prediction'] == 'Abnormal').astype(int)

# Initialize metrics list
metrics_no_colour_results = []

# Group and calculate metrics
grouped_no_colour = df_real_inference_no_colour.groupby(['Label_Set', 'Strategy'])

plt.figure(figsize=(12, 8))

for (label_set, strategy), group in grouped_no_colour:
    y_true = group['True_Binary_Num']
    y_pred = group['Pred_Binary_Num']

    TP = len(group[(group['True_Label'] == 'Abnormal') & (group['Binary_Prediction'] == 'Abnormal')])
    FN = len(group[(group['True_Label'] == 'Abnormal') & (group['Binary_Prediction'] == 'Normal')])
    TN = len(group[(group['True_Label'] == 'Normal') & (group['Binary_Prediction'] == 'Normal')])
    FP = len(group[(group['True_Label'] == 'Normal') & (group['Binary_Prediction'] == 'Abnormal')])

    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    # New Metrics
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1 = 2 * (ppv * recall) / (ppv + recall) if (ppv + recall) > 0 else 0.0

    # Corrected MCC calculation
    numerator_mcc = (TP * TN) - (FP * FN)
    denominator_mcc_product = (TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)

    if denominator_mcc_product == 0:
        mcc = 0.0
    else:
        mcc = numerator_mcc / math.sqrt(denominator_mcc_product)

    # ROC AUC
    try:
        roc_auc = roc_auc_score(y_true, y_pred)
    except ValueError:
        roc_auc = 0.0

    # Append results
    metrics_no_colour_results.append({
        'Label_Set': label_set,
        'Strategy': strategy,
        'TP': TP,
        'FN': FN,
        'TN': TN,
        'FP': FP,
        'Sensitivity': recall,
        'Specificity': specificity,
        'PPV': ppv,
        'F1_Score': f1,
        'MCC': mcc,
        'ROC_AUC': roc_auc
    })


# Output Dataframe
df_metrics_no_colour = pd.DataFrame(metrics_no_colour_results)

# Define the desired order for 'Label_Set'
desired_order = ["No Label", "L1", "L2", "L3"]

# Convert 'Label_Set' to a categorical type with the desire_no_colourr
df_metrics_no_colour['Label_Set'] = pd.Categorical(df_metrics_no_colour['Label_Set'], categories=desired_order, ordered=True)

# Sort the DataFrame by 'Label_Set'
df_metrics_no_colour = df_metrics_no_colour.sort_values('Label_Set')

print("Final Metrics Including no_colour Strategy and Additional Metrics:")
display(df_metrics_no_colour)

Final Metrics Including no_colour Strategy and Additional Metrics:


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
12,No Label,Colour + Morphology Context,176,5,230,4343,0.972376,0.050295,0.038947,0.074894,0.020015,0.511335
13,No Label,Colour + No Morphology Context,181,0,0,4573,1.000000,0.000000,0.038073,0.073354,0.000000,0.500000
14,No Label,No Colour + Morphology Context,174,7,446,4127,0.961326,0.097529,0.040456,0.077644,0.038361,0.529427
15,No Label,No Colour + No Morphology Context,0,181,4573,0,0.000000,1.000000,0.000000,0.000000,0.000000,0.500000
0,L1,Colour + Morphology Context,175,6,140,4433,0.966851,0.030614,0.037977,0.073084,-0.002811,0.498733
1,L1,Colour + No Morphology Context,181,0,0,4573,1.000000,0.000000,0.038073,0.073354,0.000000,0.500000
2,L1,No Colour + Morphology Context,167,14,601,3972,0.922652,0.131424,0.040348,0.077315,0.030836,0.527038
3,L1,No Colour + No Morphology Context,0,181,4573,0,0.000000,1.000000,0.000000,0.000000,0.000000,0.500000
4,L2,Colour + Morphology Context,180,1,26,4547,0.994475,0.005686,0.038079,0.073350,0.000409,0.500080
5,L2,Colour + No Morphology Context,181,0,0,4573,1.000000,0.000000,0.038073,0.073354,0.000000,0.500000


<Figure size 1200x800 with 0 Axes>